# LiveCodeBench run: Qwen2.5-Coder-7B-Instruct + DeepSeek-Coder-V2-Lite-Instruct + Sonnet 4.6 sample

Use this notebook on **Vast.ai, RunPod/Lambda, Colab Pro, or another CUDA GPU notebook**. Recommended hardware for the open models: NVIDIA A100 40GB or larger.

The notebook runs the official LiveCodeBench runner for the project experiments:

1. environment/GPU check;
2. official LiveCodeBench clone and install;
3. compatibility patches for DeepSeek-Coder-V2-Lite, Claude Sonnet 4.6, vLLM memory, safe API sample limits, and date-filtered self-repair;
4. no-thinking local code-generation/self-repair runs for Qwen and DeepSeek;
5. a cost-safe non-thinking Claude Sonnet 4.6 API sample;
6. collection of raw outputs plus report-ready summaries into `lcb_run_outputs.zip`.


In [ ]:
# 0. Locate workspace and run package. Works on Vast.ai (/workspace), Colab (/content), or plain Jupyter.
from pathlib import Path
import os, shutil, subprocess, sys

if Path('/workspace').exists():
    BASE_DIR = Path('/workspace')
elif Path('/content').exists():
    BASE_DIR = Path('/content')
else:
    BASE_DIR = Path.cwd()

PACKAGE_NAME = 'Nguyen_AnQuoc_LCB_Run_Package.zip'

def has_scripts(path: Path) -> bool:
    return (path / 'scripts' / '01_setup_lcb.sh').exists()

candidates = [Path.cwd(), Path.cwd().parent, BASE_DIR / 'lcb_run_package', BASE_DIR]
PACKAGE_DIR = next((p.resolve() for p in candidates if has_scripts(p)), None)

if PACKAGE_DIR is None:
    zip_candidates = [Path.cwd() / PACKAGE_NAME, BASE_DIR / PACKAGE_NAME, Path('/content') / PACKAGE_NAME]
    package_zip = next((p for p in zip_candidates if p.exists()), None)
    if package_zip is None:
        raise FileNotFoundError(f'Could not find an extracted package or {PACKAGE_NAME}. Upload the ZIP to {BASE_DIR}.')
    PACKAGE_DIR = BASE_DIR / 'lcb_run_package'
    shutil.rmtree(PACKAGE_DIR, ignore_errors=True)
    PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(package_zip), '-d', str(PACKAGE_DIR)], check=True)

LCB_DIR = BASE_DIR / 'LiveCodeBench'
HF_HOME = BASE_DIR / '.cache' / 'huggingface'
os.environ.setdefault('HF_HOME', str(HF_HOME))
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')

print('BASE_DIR    =', BASE_DIR)
print('PACKAGE_DIR =', PACKAGE_DIR)
print('LCB_DIR     =', LCB_DIR)
print('HF_HOME     =', os.environ['HF_HOME'])


In [ ]:
# 1. GPU / Python check
import subprocess, sys

try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found. Make sure the Vast.ai instance exposes an NVIDIA GPU.')

try:
    import torch
    print('Python:', sys.version)
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
except Exception as exc:
    print('Torch check skipped before setup:', repr(exc))


## 2. Clone and install LiveCodeBench

This creates a Python 3.11 `uv` environment inside the official LiveCodeBench checkout, installs dependencies, and applies the small compatibility patches included in this package.


In [ ]:
# 2. Clone LiveCodeBench if needed.
import subprocess, shutil

if not (LCB_DIR / '.git').exists():
    if LCB_DIR.exists():
        shutil.rmtree(LCB_DIR)
    subprocess.run(['git', 'clone', 'https://github.com/LiveCodeBench/LiveCodeBench.git', str(LCB_DIR)], check=True)
else:
    print('Using existing clone:', LCB_DIR)
    subprocess.run(['git', '-C', str(LCB_DIR), 'status', '--short'], check=False)


In [ ]:
# 3. Install dependencies and patch the runner.
setup_script = PACKAGE_DIR / 'scripts' / '01_setup_lcb.sh'
subprocess.run(['bash', str(setup_script)], cwd=str(LCB_DIR), check=True)


In [ ]:
# 4. Verify model entries and patch state.
subprocess.run([str(LCB_DIR / '.venv' / 'bin' / 'python'), str(PACKAGE_DIR / 'scripts' / '02_patch_deepseek_v2_lite.py'), str(LCB_DIR)], check=True)
subprocess.run([str(LCB_DIR / '.venv' / 'bin' / 'python'), str(PACKAGE_DIR / 'scripts' / '02b_patch_vllm_max_model_len.py'), str(LCB_DIR)], check=True)
subprocess.run([str(LCB_DIR / '.venv' / 'bin' / 'python'), str(PACKAGE_DIR / 'scripts' / '02c_patch_selfrepair_date_filter.py'), str(LCB_DIR)], check=True)
subprocess.run([str(LCB_DIR / '.venv' / 'bin' / 'python'), str(PACKAGE_DIR / 'scripts' / '02d_patch_claude_sonnet_4_6.py'), str(LCB_DIR)], check=True)
subprocess.run([str(LCB_DIR / '.venv' / 'bin' / 'python'), str(PACKAGE_DIR / 'scripts' / '02e_patch_debug_sample_limit.py'), str(LCB_DIR)], check=True)
subprocess.run(['grep', '-n', 'Qwen2.5-Coder-7B-Instruct\|DeepSeek-Coder-V2-Lite-Instruct\|claude-sonnet-4-6', str(LCB_DIR / 'lcb_runner' / 'lm_styles.py')], check=False)


## 3. Choose run settings

For a quick setup check, keep `SMOKE=1`. For final report results, set `SMOKE=0` and rerun the code-generation and self-repair cells.

The default proposal-aligned window is `release_v5`, `2024-08-01` through `2025-01-31`, `n=1`, `temperature=0.2`.


In [ ]:
# 5. Shared run configuration.
import os

os.environ['RELEASE_VERSION'] = 'release_v5'
os.environ['START_DATE'] = '2024-08-01'
os.environ['END_DATE'] = '2025-01-31'
os.environ['N'] = '1'
os.environ['TEMPERATURE'] = '0.2'
os.environ['TOP_P'] = '0.95'
os.environ['MAX_TOKENS'] = '4096'
os.environ['TIMEOUT'] = '10'
os.environ['NUM_PROCESS_EVALUATE'] = '4'
os.environ['TENSOR_PARALLEL_SIZE'] = '1'
os.environ['DTYPE'] = 'bfloat16'
os.environ['LCB_MAX_MODEL_LEN'] = '8192'
os.environ['VLLM_GPU_MEMORY_UTILIZATION'] = '0.90'
os.environ['CACHE_BATCH_SIZE'] = '25'
os.environ['API_SAMPLE_SIZE'] = '3'
os.environ['SONNET_MAX_TOKENS'] = '2048'
os.environ['ANTHROPIC_MULTIPROCESS'] = '1'

# Keep this as '1' for the first run. Change to '0' for final report results.
os.environ['SMOKE'] = '1'

keys = ['RELEASE_VERSION','START_DATE','END_DATE','N','TEMPERATURE','MAX_TOKENS','LCB_MAX_MODEL_LEN','SMOKE','API_SAMPLE_SIZE','SONNET_MAX_TOKENS']
print({k: os.environ[k] for k in keys})


## 4. Smoke tests

Run these with `SMOKE=1`. They use LiveCodeBench debug mode, which selects 15 benchmark instances from the filtered date window.


In [ ]:
# 6. Qwen code-generation smoke test.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '03_run_codegen_qwen.sh')], cwd=str(LCB_DIR), check=True)


In [ ]:
# 7. DeepSeek code-generation smoke test.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '04_run_codegen_deepseek.sh')], cwd=str(LCB_DIR), check=True)


In [ ]:
# 8. Optional self-repair smoke tests. Run after the two code-generation smoke tests above.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '05_run_selfrepair_qwen.sh')], cwd=str(LCB_DIR), check=True)
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '06_run_selfrepair_deepseek.sh')], cwd=str(LCB_DIR), check=True)


## 5. Safe Sonnet 4.6 API sample

These cells use `claude-sonnet-4-6` in LiveCodeBench's normal Claude style, not extended-thinking mode. They always run `--debug` with `LCB_DEBUG_LIMIT=API_SAMPLE_SIZE`, so the default is only three problems.

Set `ANTHROPIC_KEY` or `ANTHROPIC_API_KEY` first. The self-repair sample is optional and should be run only after the Sonnet code-generation sample succeeds.


In [ ]:
# 9. Configure Anthropic key for the safe Sonnet sample.
# Prefer setting this as an environment variable in your notebook/server instead of hardcoding it here.
# import os
# os.environ['ANTHROPIC_KEY'] = 'sk-ant-...'

import os
has_key = bool(os.environ.get('ANTHROPIC_KEY') or os.environ.get('ANTHROPIC_API_KEY'))
print('Anthropic key configured:', has_key)
print('Sonnet sample size:', os.environ.get('API_SAMPLE_SIZE', '3'))
print('Sonnet max tokens:', os.environ.get('SONNET_MAX_TOKENS', '2048'))


In [ ]:
# 10. Claude Sonnet 4.6 code-generation sample. This is intentionally sample-limited.
if not (os.environ.get('ANTHROPIC_KEY') or os.environ.get('ANTHROPIC_API_KEY')):
    raise RuntimeError('Set ANTHROPIC_KEY or ANTHROPIC_API_KEY before running the Sonnet sample.')
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '10_run_codegen_sonnet_sample.sh')], cwd=str(LCB_DIR), check=True)


In [ ]:
# 11. Optional Claude Sonnet 4.6 self-repair sample. Run only after the Sonnet code-generation sample.
if not (os.environ.get('ANTHROPIC_KEY') or os.environ.get('ANTHROPIC_API_KEY')):
    raise RuntimeError('Set ANTHROPIC_KEY or ANTHROPIC_API_KEY before running the Sonnet sample.')
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '11_run_selfrepair_sonnet_sample.sh')], cwd=str(LCB_DIR), check=True)


## 6. Actual final-report runs

Set `SMOKE=0`, then run the two code-generation cells and the two self-repair cells. These are the results to use in the report.


In [ ]:
# 12. Switch to full filtered run.
os.environ['SMOKE'] = '0'
print('SMOKE =', os.environ['SMOKE'])


In [ ]:
# 13. Qwen full code-generation run.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '03_run_codegen_qwen.sh')], cwd=str(LCB_DIR), check=True)


In [ ]:
# 14. DeepSeek full code-generation run.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '04_run_codegen_deepseek.sh')], cwd=str(LCB_DIR), check=True)


In [ ]:
# 15. Qwen full self-repair run.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '05_run_selfrepair_qwen.sh')], cwd=str(LCB_DIR), check=True)


In [ ]:
# 16. DeepSeek full self-repair run.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '06_run_selfrepair_deepseek.sh')], cwd=str(LCB_DIR), check=True)


## 7. Summarize and collect outputs

The ZIP contains raw LiveCodeBench outputs, overall summaries, per-instance rows, difficulty/platform/month breakdowns, and self-repair gains. Upload `lcb_run_outputs.zip` back so the report tables and discussion can be finalized.


In [ ]:
# 17. Summarize and zip results.
subprocess.run(['bash', str(PACKAGE_DIR / 'scripts' / '08_collect_results.sh')], cwd=str(LCB_DIR), check=True)
zip_path = LCB_DIR / 'lcb_run_outputs.zip'
print('Created:', zip_path)
print('Size:', zip_path.stat().st_size, 'bytes')

# Also copy to the top-level workspace for easier download in Vast.ai/Jupyter.
workspace_zip = BASE_DIR / 'lcb_run_outputs.zip'
shutil.copy2(zip_path, workspace_zip)
print('Copied to:', workspace_zip)


In [ ]:
# 18. Preview summary tables.
summary_md = LCB_DIR / 'run_summaries' / 'lcb_summary.md'
if summary_md.exists():
    print(summary_md.read_text()[:4000])
else:
    print('Summary not found yet.')
